# Train the Corvid chess model on Colab

This notebook trains Corvid's grandmaster-move model on a free Colab GPU:

1. **Phase 1** – Chess.com games of grandmasters rated 2500+ (no bullet), from the Kaggle dataset `dimitrioskourtikakis/gm-games-chesscom`.
2. **Phase 2** – fine-tuning on Lichess broadcast games (over-the-board tournaments, Jan 2020 – Sep 2026), at a lower learning rate.
3. **Export** to `corvid.onnx`, which runs in the browser with ONNX Runtime Web.

## How to run

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** Allow Google Drive access when asked. Progress and checkpoints are saved to `My Drive/corvid`.
3. Keep this tab open and the screen on. On an iPad, Colab disconnects if Safari goes to the background for long.

**If it disconnects:** reconnect and choose **Run all** again. Every step skips work that is already done and training resumes from the last checkpoint (saved every 2 minutes).

Rough timings on a T4: data prep 20–30 min (first run only), phase 1 about 40–60 min, phase 2 about 10–20 min.

## 1. Setup

In [ ]:
import os, subprocess, torch

def sh(cmd):
    """Run a shell command and stream its output into the notebook."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU! Use Runtime → Change runtime type → T4 GPU, then Run all again.")

from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/corvid"      # saved between sessions
LOCAL = "/content/corvid-local"             # fast local disk, lost on disconnect
os.makedirs(f"{WORK}/checkpoints", exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)

In [ ]:
%cd /content
sh("rm -rf corvid-chess-engine && git clone -q -b training-pipeline https://github.com/Studio-Roo-Official/corvid-chess-engine")
sh("pip -q install chess zstandard kagglehub onnx onnxruntime")
%cd /content/corvid-chess-engine/training

## 2. Chess.com data

Downloads the 3 GB Kaggle dataset (15 GB unpacked) and keeps positions where the grandmaster is to move, rated 2500+, in games of 3 minutes or longer. A random 25% of those games are used (about 25 million positions). The result (1.7 GB) is saved to Drive, so this only runs once.

In [ ]:
def restore(name):
    """Copy a finished data folder from Drive to local disk. Returns False if it isn't on Drive yet."""
    if not os.path.exists(f"{WORK}/{name}/.done"):
        return False
    if not os.path.exists(f"{LOCAL}/{name}/.done"):
        print(f"copying {name} from Drive…")
        sh(f"rm -rf {LOCAL}/{name} && cp -r {WORK}/{name} {LOCAL}/{name}")
    return True

if not restore("data"):
    import kagglehub
    path = kagglehub.dataset_download("dimitrioskourtikakis/gm-games-chesscom")
    sh(f"python prepare_data.py '{path}/GM_games_dataset.csv' --out {LOCAL}/data --sample 0.25 --workers {os.cpu_count()}")
    sh(f"touch {LOCAL}/data/.done && rm -rf {WORK}/data && cp -r {LOCAL}/data {WORK}/data")
    sh("rm -rf /root/.cache/kagglehub")   # free 18 GB of disk
print("Chess.com data ready:", sorted(os.listdir(f"{LOCAL}/data")))

## 3. Lichess broadcast data

Downloads the 81 monthly broadcast files (0.7 GB) and keeps moves by players rated 2500+, dropping games that were broadcast more than once.

In [ ]:
if not restore("data_broadcast"):
    os.makedirs(f"{LOCAL}/broadcast", exist_ok=True)
    months = [f"{y}-{m:02d}" for y in range(2020, 2027) for m in range(1, 13) if (y, m) <= (2026, 9)]
    with open(f"{LOCAL}/broadcast_urls.txt", "w") as fh:
        fh.write("\n".join(f"https://database.lichess.org/broadcast/lichess_db_broadcast_{x}.pgn.zst" for x in months) + "\n")
    sh(f"cd {LOCAL}/broadcast && xargs -n1 -P4 curl -sSfLO --retry 3 < ../broadcast_urls.txt")
    sh(f"python prepare_broadcast.py {LOCAL}/broadcast/*.pgn.zst --out {LOCAL}/data_broadcast --workers {os.cpu_count()}")
    sh(f"touch {LOCAL}/data_broadcast/.done && rm -rf {WORK}/data_broadcast && cp -r {LOCAL}/data_broadcast {WORK}/data_broadcast")
print("Broadcast data ready:", sorted(os.listdir(f"{LOCAL}/data_broadcast")))

## 4. Phase 1 – train on Chess.com games

Two passes over the data. Watch `val acc` (how often the model picks the grandmaster's exact move on games it never trained on) and `top5`.

In [ ]:
sh(f"python train.py --data {LOCAL}/data --ckpt {WORK}/checkpoints/corvid.pt "
   f"--epochs 2 --bs 1024 --lr 2e-3 --save-every 120 --eval-every 2000 --threads {os.cpu_count()}")

## 5. Phase 2 – fine-tune on Lichess broadcast games

Starts from the phase 1 model with a lower learning rate. `extra val` is the Chess.com check, so you can see the model isn't forgetting phase 1.

In [ ]:
sh(f"python train.py --data {LOCAL}/data_broadcast --ckpt {WORK}/checkpoints/corvid_broadcast.pt "
   f"--init-from {WORK}/checkpoints/corvid.pt --extra-val {LOCAL}/data "
   f"--epochs 2 --bs 1024 --lr 3e-4 --warmup 300 --save-every 120 --eval-every 1000 --threads {os.cpu_count()}")

## 6. Export and try it

In [ ]:
sh(f"python export_onnx.py --ckpt {WORK}/checkpoints/corvid_broadcast.pt --out {WORK}/corvid.onnx")

import chess, numpy as np, onnxruntime as ort, torch
from model import planes_from_packed
from prepare_data import encode

sess = ort.InferenceSession(f"{WORK}/corvid.onnx")

def top_moves(fen, k=5):
    board = chess.Board(fen)
    white = board.turn == chess.WHITE
    arr, castle, ep = encode(board, white)
    planes = planes_from_packed(torch.from_numpy(arr[None]), torch.tensor([[castle, ep]], dtype=torch.uint8)).numpy()
    policy, value = sess.run(None, {"planes": planes})
    scores = {}
    for mv in board.legal_moves:
        f, t = (mv.from_square, mv.to_square) if white else (mv.from_square ^ 56, mv.to_square ^ 56)
        if mv.promotion in (None, chess.QUEEN):
            scores[mv] = policy[0, f * 64 + t]
    s = np.array(list(scores.values())); p = np.exp(s - s.max()); p /= p.sum()
    best = sorted(zip(scores, p), key=lambda x: -x[1])[:k]
    print(fen)
    print("  value for side to move:", round(float(value[0]), 3))
    for mv, pr in best:
        print(f"  {board.san(mv):8s} {pr:.1%}")

top_moves(chess.STARTING_FEN)
top_moves("r1bqkbnr/pppp1ppp/2n5/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 2 3")

In [ ]:
from google.colab import files
files.download(f"{WORK}/corvid.onnx")   # also saved in My Drive/corvid/corvid.onnx